# 内部机制 1：签名编译管线——声明如何变成签名计划

主教程线教你用 PLLM 做观测、定制和训练；这条三篇的**内部机制支线**换一个视角：不再写新的负载，而是拆开框架内部，看它凭什么兑现主线里那些保证。本篇回答一个具体问题：**一个 Python 类的声明，如何一步步变成一份签名的、不可篡改的执行计划？**

为什么值得花这一篇的时间？你在主线里见过的每一条运行时保证——请求之间的状态隔离、状态字节预算、KV 回滚与重放——都不是运行时临时决策的结果，而是编译期就被推导出来、写进计划、盖上签名的性质；运行时只负责校验和执行。看懂编译管线，就看懂了这些保证的出处。

本篇全程在 CPU 上运行，不加载模型、不构造 vLLM 引擎。这不是妥协，而是管线的本来面貌：编译是一次纯数据变换——Python 侧的声明对象和后端事实进去，Rust 调度器的签名计划出来，整个过程不碰任何模型权重。我们手工扮演引擎的角色，构造一份编译事实，把真实管线走一遍。你在主线里用 `make_llm` 挂上负载时，引擎侧走的就是这条管线的真实版本。

整条管线四步，也是本篇的路线图：

1. **声明**：负载类被冻结、扫描，产出语义图——任务、状态、效果、时机窗口；
2. **事实**：后端申报自己的指纹、能力、时间轴与预算；
3. **编译**：Rust 联合求解实现、布局、锚点与放置，产出签名的物理计划；
4. **绑定**：worker 用同一份事实复核签名计划，物化出可执行的结构。

## 声明侧：负载说了什么，没说什么

样例用内置负载 InPlaceTTT（原位 test-time training）：它每步对选中的模型权重做一次小步更新，并把结果以参数 overlay 的形式应用回模型。你不需要会写它——读过主线 00_quickstart 就够了；这里只把它当作「一个已经写好的负载类」。

一个负载的**声明**包含四类东西：**算什么**（任务）、**什么时候算**（锚点窗口）、**动哪些状态**（效果）、**有哪些实现可选**。它**不**声明执行顺序、张量布局、内存摆放——那些是编译器的事。

声明只是编译的一半输入；另一半是**编译事实** `BackendCompileFacts`：后端探测（probe，含后端 ABI、runner、模型三枚指纹与设备/并行度/形状上限）、时间轴（锚点序列与能力清单）、页提供者与状态预算。真实部署里这些由 vLLM 附着侧探测生成；今天我们手工构造一份，值都取得很小、指纹用占位字符串，只为把类型走通。

下面这段代码先构造事实，再调用 `declare`——负载类被编译成一张**语义图**（`SemanticGraph`），外加布局请求和附着规格。注意：此刻还没有任何执行计划。

In [1]:
import torch

from pllm.authoring import BackendCompileFacts, ParameterTargetShape
from pllm.contracts.backend_attachment import (
    AttachmentGraphMode, AttachmentTopology, BackendPlacementProbe)
from pllm.contracts.parameter_overlay import ParameterModuleKind, ParameterRole, ParameterTarget
from pllm.contracts.state_representation import StateDeviceKind
from pllm.contracts.tensor_layout import TensorDType
from pllm.execution.ptts.page_bundle_planning import PageBundleProviderFacts
from pllm.execution.runtime_facts import RuntimeFacts
from pllm.workloads.inplace_ttt import InPlaceTTTConfig, InPlaceTTTWorkload

# overlay 目标：第 0、1 层 MLP 的 down projection 权重
targets = tuple(
    ParameterTarget(layer, ParameterModuleKind.MLP, ParameterRole.DOWN_PROJECTION_WEIGHT)
    for layer in (0, 1))
config = InPlaceTTTConfig(targets, chunk_size=2, learning_rate=0.3)


def make_facts(capabilities=("transition/full_root@1",)):
    """手工构造一份 CPU 编译事实；真实部署里由 vLLM 附着侧探测生成。"""
    probe = BackendPlacementProbe(
        "tutorial.inplace_ttt@1",          # 适配器标识
        "a" * 64, "b" * 64, "c" * 64,      # 后端 ABI / runner / 模型三枚指纹（占位值）
        AttachmentTopology.IN_PROCESS,     # 进程内附着
        AttachmentGraphMode.EAGER,         # 不做 CUDA graph 捕获
        StateDeviceKind.CPU, 0,            # 设备：CPU，序号 0
        1, 1,                              # 张量并行 / 流水并行度
        TensorDType.FLOAT64, 3, 8,         # dtype、hidden 大小、单批最大打包行数
        (),                                # 本负载不读 hidden
        capabilities)
    timeline = RuntimeFacts(torch.device("cpu")).timeline(capabilities)
    providers = (PageBundleProviderFacts("test.cpu.full_root@1", 4),)
    return BackendCompileFacts(
        probe, timeline, providers,
        2,        # 最多同时准入 2 条请求 lineage
        4096,     # 托管状态字节预算
        0,        # 初始化种子
        tuple(ParameterTargetShape(target, 3, 2) for target in targets))


def short_impl(implementation_id: str) -> str:
    """python:<指纹>:<变体> 形态的 ID 只留指纹前 16 位。"""
    if implementation_id.startswith("python:"):
        _, fingerprint, variant = implementation_id.split(":")
        return f"python:{fingerprint[:16]}...:{variant}"
    return implementation_id


graph, layouts, spec = InPlaceTTTWorkload(config).declare(make_facts())

print(f"负载: {graph.workload_id}   声明参数摘要: {graph.execution_parameters_digest[:16]}...")
print(f"附着类型: {spec.kind.value}   生效锚点: {spec.sampler_anchor}")
for task in graph.tasks:
    effects = ", ".join(f"{e.kind.value}({e.resource})" for e in task.effects)
    print(f"任务 {task.symbol:<11} 窗口 [{task.timing.after} -> {task.timing.before}]   效果: {effects}")
    print(f"    实现候选: {[short_impl(i.implementation_id) for i in task.implementations]}")
for state in graph.states:
    print(f"状态 {state.resource}   作用域: {state.scope.value}")

负载: inplace_ttt.dense_fast_weight   声明参数摘要: 653e939654437599...
附着类型: parameter_overlay   生效锚点: sampler.complete
任务 publish     窗口 [state.publish -> next.prepare]   效果: publish(dense_fast_weights)
    实现候选: ['canonical.state:cuda_publish_parameter_overlay@1', 'canonical.state:publish_parameter_overlay@1']
任务 transition  窗口 [model.execute -> state.publish]   效果: transition_maybe(dense_fast_weights)
    实现候选: ['python:b2e9ed6c744740ed...:cpu', 'python:b2e9ed6c744740ed...:cuda']
状态 dense_fast_weights   作用域: request


输出的信息量很大，挑三个最值得记住的点：

- **两类任务**。`transition` 是负载自己的数学，实现 ID 形如 `python:<64 位指纹>:cpu`——同一枚指纹带 cpu/cuda 两个候选变体。指纹是所创作数学的冷结构签名（字节码加闭包传递闭包，见术语表「指纹」）：改一行数学，指纹就变，整条链重新签名；装载从不重新执行。`publish` 是框架自己的 canonical 任务，实现固定为 `canonical.state:publish_parameter_overlay@1`——它的职责是发布私有迁移，使其原子可见。
- **声明顺序不是执行顺序**。声明里 `publish` 排在 `transition` 前面，但它反而后执行——顺序由时机窗口与效果推导，与书写顺序无关。后面编译输出里会看到编译器给出的真实顺序。
- **效果与窗口是全部约束**。`transition_maybe`（可能写）与 `publish`（发布）限定了谁能动 `dense_fast_weights` 状态；窗口 `[model.execute, state.publish]` 与 `[state.publish, next.prepare]` 限定了何时动。附着规格则声明 overlay 在 `sampler.complete` 锚点生效。

你可能还注意到那枚「声明参数摘要」——config 里的每个字段都被签了进去，后面会看到改动它的后果。

### 事实侧：成本事实与能力门控

签名计划要求每个任务、每个输出都有资源事实——内存受限的调度需要完整的成本视图。但用户只给自己的数学估价，从不给框架的 import/publish 管道估价：这些 canonical 任务的拷贝直接落在已分配的私有页里，所以 `derive_framework_resources` 在编译前为它们补零成本事实。

能力清单同样不是摆设：`publish` 的 canonical 实现要求 `transition/full_root@1` 能力。下面先看补全前后的对比，再故意删掉这个能力，看编译如何反应。

In [2]:
from pllm.authoring import derive_framework_resources

facts = make_facts()
raw, derived = facts.timeline, derive_framework_resources(graph, facts.timeline)

print("后端原始申报:")
print(f"  任务成本事实: {list(raw.task_resources)}")
print(f"  值成本事实:   {list(raw.value_resources)}")
print("框架补全后:")
for fact in derived.task_resources:
    print(f"  任务 {fact.task}: workspace={fact.workspace_bytes}B, resident={fact.resident_bytes}B")
for fact in derived.value_resources:
    print(f"  值 {fact.task}.{fact.output}: {fact.bytes}B")

# 删掉 publish 实现要求的 transition/full_root@1 能力，再编译一次
try:
    InPlaceTTTWorkload(config).compile_backend(make_facts(capabilities=()))
    print("意外：编译成功了")
except ValueError as error:
    print(f"失败即坏: {error}")

后端原始申报:
  任务成本事实: []
  值成本事实:   []
框架补全后:
  任务 publish: workspace=0B, resident=0B
  值 publish.publication: 0B
失败即坏: no joint state representation and AER placement fits versions, timing and capacity


对比说明了分工：这条时间轴上后端对任务成本的原始申报是空的（本例没有给用户数学报成本），框架只为自己的 canonical `publish` 任务补上两条零成本事实——任务 workspace 0B、输出 `publication` 0B。而删掉 `transition/full_root@1` 之后，编译直接以「找不到同时满足版本、时机与容量的状态表示与放置」收场：**能力是签名输入的一部分，缺了就失败即坏，不存在降级执行**。

## 编译：联合求解与签名

现在把声明和事实拼起来。`compile_backend` 是两步的复合：

- `plan_backend`：语义图、补全后的时间轴、页提供者和预算一起交给 Rust 调度器，由它**联合**选择实现、执行区域、锚点、执行域与状态表示，产出 `PlanBundle` 并签名；
- `bind_backend`：拿同一份事实复核这份签名附着、检查容量，物化 PageBundleArena，并把数学绑到选中的实现上。

下面把两步拆开走一遍，然后翻看计划里最说明问题的几个字段。

In [3]:
workload = InPlaceTTTWorkload(config)
attachment = workload.plan_backend(facts)              # 语义图 + 事实 -> 签名计划与签名附着
deployment = workload.bind_backend(facts, attachment)  # 复核事实，物化 arena，绑定数学
plan = deployment.execution.arena.plan

decision = plan.schedule_decision
print(f"调度签名:   {decision.signature}")
print(f"图摘要:     {decision.graph_digest[:24]}...")
print(f"时间轴摘要: {decision.timeline_digest[:24]}...")
print(f"状态摘要:   {decision.state_plan_digest[:24]}...")
print(f"回退任务数: {decision.fallback_count}   峰值内存: {decision.peak_memory_bytes}B")

anchor_name = {anchor.ordinal: anchor.symbol for anchor in plan.timeline.anchors}
print("物理任务（执行顺序由编译器推导，与声明顺序无关）:")
for task in plan.schedule_physical.tasks:
    print(f"  id={task.task_id}  锚点 {anchor_name[task.anchor_id]:<15} "
          f"实现 {short_impl(task.implementation_id)}   释放值 {list(task.release_values)}")

调度签名:   e38a26afe31b61fd2396c555499ab3a55e17a9cd2aaf64f51201b41f9672f7e3
图摘要:     e92d889755423193b1ecea9e...
时间轴摘要: cd6e9e85581d5f05c8cb5ede...
状态摘要:   c6cf20de06beb7728f2d3461...
回退任务数: 0   峰值内存: 880B
物理任务（执行顺序由编译器推导，与声明顺序无关）:
  id=1  锚点 model.execute   实现 python:b2e9ed6c744740ed...:cpu   释放值 []
  id=0  锚点 state.publish   实现 canonical.state:publish_parameter_overlay@1   释放值 [1]


输出里四个字段值得记住：

- **调度签名与三枚摘要**：`signature` 锚定整个调度决策；`graph_digest`、`timeline_digest`、`state_plan_digest` 分别锚定语义图、时间轴、状态计划三份输入——任何一份变了，签名即失效；
- **回退任务数恒为 0**：回退指签名原语之外的执行，契约上不允许存在；
- **峰值内存 880B**：调度器在预算内证明过的驻留上界，是编译期算出来的，不是运行后测出来的；
- **释放值**：`publish` 任务末尾的 `释放值 [1]` 是 Rust 签名的最后使用释放——每个值的最后一次使用都写在计划里，执行器只是照做，没有第二个回收策略。

物理任务的真实顺序也印证了前面的说法：`transition` 在 `model.execute` 锚点先跑，`publish` 在 `state.publish` 锚点后跑——与声明顺序恰好相反。

### 封闭编码：计划的字节形态

计划要跨越进程边界送到 worker（内部机制第 3 篇的主题），所以它必须能编成封闭字节：`encode_boundary` 产出不含任何活引用的 JSON，九个顶层字段把语义图、时间轴、布局、状态与调度决策全部摊开；`decode_boundary` 解码回来与原件逐字段相等。

签名链路有两个可以直接动手验证的性质：**确定性**——同样的声明加同样的事实，重编译逐字节相同；**敏感性**——改动任何输入，哪怕只是 config 里的一个学习率，摘要与签名全部更换。「不可篡改」因此不是口号：篡改一定留下不同的哈希，而校验方只认签名。

In [4]:
import json
from dataclasses import replace

from pllm.contracts.compilation import PlanBundle
from pllm.execution.codec import decode_boundary, encode_boundary

blob = encode_boundary(plan)
restored = decode_boundary(blob, PlanBundle)
print(f"封闭编码: {len(blob)} 字节 JSON, 解码回等值对象: {restored == plan}")
print(f"顶层字段: {sorted(json.loads(blob))}")

again = workload.compile_backend(facts).execution.arena.plan
print(f"同样输入重编译，计划逐字节相同: {again == plan}")

changed_plan = InPlaceTTTWorkload(replace(config, learning_rate=0.5)).compile_backend(
    make_facts()).execution.arena.plan
print("learning_rate 0.3 -> 0.5 之后:")
print(f"  声明参数摘要相同: {changed_plan.semantic_graph.execution_parameters_digest == graph.execution_parameters_digest}")
print(f"  图摘要相同:       {changed_plan.schedule_decision.graph_digest == decision.graph_digest}")
print(f"  调度签名相同:     {changed_plan.schedule_decision.signature == decision.signature}")

封闭编码: 8139 字节 JSON, 解码回等值对象: True
顶层字段: ['schedule_decision', 'schedule_physical', 'semantic_graph', 'state_decision', 'state_physical', 'state_request', 'tensor_layout_decision', 'tensor_layout_request', 'timeline']
同样输入重编译，计划逐字节相同: True
learning_rate 0.3 -> 0.5 之后:
  声明参数摘要相同: False
  图摘要相同:       False
  调度签名相同:     False


三个 False 说明学习率一路进了声明参数摘要、图摘要与调度签名——config 本身就是签名的一部分。

最后一步也是管线的出口。`bind_backend` 是 worker 侧复核签名附着的入口，它**不重跑优化**，只做三件事：probe 逐字段比对、时间轴补全后与签名计划比对、容量不小于签名额度。不一致就拒绝——worker 校验的是固定见证，不是重新规划。

下面故意用一份被「调包」的事实（时间轴少了能力申报）去绑定刚才那份签名附着。

In [5]:
tampered = replace(facts, timeline=RuntimeFacts(torch.device("cpu")).timeline())
try:
    workload.bind_backend(tampered, attachment)
    print("意外：绑定成功了")
except ValueError as error:
    print(f"绑定被拒绝: {error}")

绑定被拒绝: worker backend facts differ from the original signed attachment


## 小结

回看整条管线：负载类被冻结成**声明**（语义图），后端申报**事实**（probe、时间轴、能力、预算），框架为 canonical 任务补齐零成本事实后，Rust 调度器把二者钉成一份**签名的物理计划**，worker 再用同一份事实复核并物化执行结构。

读完本篇你应该建立的心智模型是：**声明决定「算什么」，事实决定「在哪算、有什么能力」，编译器把二者钉成一份签名计划；此后的每个环节——worker 绑定、进程间传输、运行时执行——只校验这份计划，不重新发明它**。隔离、预算、回放这些运行时保证，全都是这份签名计划上可推导的性质。

下一篇《内部机制 2：托管状态与页 arena》拆开计划里那 880B 背后的记账系统：状态如何分页、版本如何被 pin 住与释放、写如何走 COW，以及预算由谁守门。